In [1]:
%run common_imports.py

%matplotlib qt
%config InlineBackend.figure_format = 'retina'
sns.set_context("talk")

%reload_ext autoreload
%autoreload 2
pd.options.display.max_rows = 600
pd.set_option('display.float_format', lambda x: '%.9f' % x)

dj.config['display.limit'] = 10**3  

os.environ["SPYGLASS_USE_TRANSACTIONS"] = "1"  
os.environ['KACHERY_API_KEY'] = "RhysjLwgmBAt2ObCyXXaDnqAv2kTdYRa"

[2026-03-12 19:50:46,100][INFO]: DataJoint is configured from /media/labuser/NA_1_2025/spyglass/wilbur/dj_local_conf.json
[2026-03-12 19:50:46,541][INFO]: DataJoint 0.14.9 connected to anirudh@172.16.102.154:3306


In [2]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats         
from scipy.stats import chi2 
import spyglass.linearization.v1 as sgpl                                                                       

### Load data

In [3]:
#extract position data
#read from csv
trialized_position = pd.read_csv("/media/labuser/NA_1_2025/spyglass/wilbur/analysis/position/trialized_position.csv", index_col = "time")

In [4]:
#extract spikes
#read from npz
data = np.load("/media/labuser/NA_1_2025/spyglass/wilbur/analysis/final_spikes/mfpc_spikes.npz", allow_pickle=True)
mpfc_spikes = [data[f"arr_{i}"] for i in range(len(data.files))]


### Prepare dataframe for regression

In [17]:
def fit_glm_all_units(formula: str,
                      cov_df: pd.DataFrame,
                      spike_counts_masked: np.array,
                      unit_ids: np.array,
                      bin_size = 0.002):
    
    rows = []
    for i, uid in enumerate(unit_ids):
        df = cov_df.copy()
        df["spike_count"] = spike_counts_masked[i]  # pre-masked counts
        try:
            res = smf.glm(formula, data=df, family=sm.families.Poisson()).fit(disp=False)
            rows.append(dict(                                                                                       
                unit=uid,   
                aic=res.aic,
                llf=res.llf,
                deviance=res.deviance,
                n_params=len(res.params),
                n_obs=int(res.nobs),
                converged=res.converged,
                coef=res.params.to_dict(),
                bse=res.bse.to_dict(),
                deviance_null = res.null_deviance,
                df_model = res.df_model
            ))

        except Exception as e:
            rows.append(dict(
                unit=uid, aic=np.nan, llf=np.nan, deviance=np.nan,
                n_params=np.nan, n_obs=np.nan, converged=False,
                coef=None, bse=None, deviance_null=np.nan,
                df_model=np.nan, error=str(e)          
            ))


    return pd.DataFrame(rows)

In [18]:
BIN_SIZE = 0.002  

bin_edges = np.arange(
    trialized_position.index.min(), trialized_position.index.max() + BIN_SIZE, BIN_SIZE
)
bin_centers = bin_edges[:-1] + BIN_SIZE / 2

spike_counts = np.array([np.histogram(spikes, bins=bin_edges)[0] for spikes in mpfc_spikes]) 

In [19]:
def interp_col(col_values, times, bin_centers):
    if pd.api.types.is_numeric_dtype(col_values):
        vals = col_values.astype(float)
        valid = ~np.isnan(vals)
        if valid.sum() < 2:
            return np.full(len(bin_centers), np.nan)
        # Exclude NaN anchor points — np.interp propagates NaN from any bracketing point
        return np.interp(bin_centers, times[valid], vals[valid], left=np.nan, right=np.nan)
    else:
        idx = np.searchsorted(times, bin_centers).clip(0, len(times) - 1)
        return col_values.iloc[idx].values

cols_to_interp = [c for c in trialized_position.columns if c != "video_frame_ind"]
times = trialized_position.index.astype(float).values

interpolated = {col: interp_col(trialized_position[col], times, bin_centers) for col in cols_to_interp}

interp_trialised_position = pd.DataFrame(interpolated, columns=cols_to_interp)
interp_trialised_position.insert(0, "time_bin_center", bin_centers)

mask = (interp_trialised_position["zone"]=="run") &\
    (interp_trialised_position["trial_type"].isin(["outbound", "inbound"]))


cov_df = interp_trialised_position[mask]
spike_counts_masked = spike_counts[:, mask]
unit_ids = np.arange(0, len(spike_counts_masked))

# print(cov_df.head(1))
# print(spike_counts_masked[0].shape)
#print(unit_ids)

In [20]:
cov_df = cov_df.rename(columns={"left/right": "choice"})

In [21]:
# Intersection of all predictor filters — all models fit on this for valid AIC comparison
common_mask = (
    cov_df["speed"].notna() & (cov_df["speed"] > 5) & (cov_df["speed"] < 120)
    & cov_df["linear_position"].notna()
)
cov_df_common = cov_df[common_mask].copy()
spike_counts_common = spike_counts_masked[:, common_mask]

# Scaling params defined once — used by all single-unit and all-units cells
speed_min_val = cov_df_common["speed"].min()
speed_max_val = cov_df_common["speed"].max()
pos_min_val   = cov_df_common["linear_position"].min()
pos_max_val   = cov_df_common["linear_position"].max()

cov_df_common["speed_scaled"] = (cov_df_common["speed"] - speed_min_val) / (speed_max_val - speed_min_val)
cov_df_common["pos_scaled"]   = (cov_df_common["linear_position"] - pos_min_val) / (pos_max_val - pos_min_val)

# Outbound-only common subset — baseline and choice model
outbound_common_mask = common_mask & (cov_df["trial_type"] == "outbound")
cov_df_out_common = cov_df[outbound_common_mask].copy()
spike_counts_out_common = spike_counts_masked[:, outbound_common_mask]
cov_df_out_common["speed_scaled"] = (cov_df_out_common["speed"] - speed_min_val) / (speed_max_val - speed_min_val)
cov_df_out_common["pos_scaled"]   = (cov_df_out_common["linear_position"] - pos_min_val) / (pos_max_val - pos_min_val)

print(f"cov_df_common: {len(cov_df_common):,} bins")
print(f"cov_df_out_common: {len(cov_df_out_common):,} bins (outbound only)")

cov_df_common: 1,756,595 bins
cov_df_out_common: 816,658 bins (outbound only)


### Models:

#### Single variable models:
1. Null model (constant rate)
2. Null model (outbound only, for comparison with other outbound-only models)
2. spike_count ~ trial_type (categorical)
3. spike_count ~ left/right choice (categorical)
4. spike_count ~ speed (linear)
5. spike_count ~ bs(speed, df = 4) (spline)
6. spike_count ~ bs(linear_position, df = 8) (spline)
7. **spike_count ~ cr(progress, df= 6) (spline)**

#### Mutli-variable models:
1. spike_count ~ trial_type + bs(speed) + bs(linear_position) + bs(trial_progress)


## Single-variable models

### Null model

#### Fit on one unit 

In [10]:
unit_idx = 9
spk_cov_df = cov_df.copy()
spk_cov_df["spike_count"] = spike_counts_masked[unit_idx]

spk_cov_df_common = cov_df_common.copy()
spk_cov_df_common["spike_count"] = spike_counts_common[unit_idx]

In [11]:
model_constant = smf.glm("spike_count ~ 1", data=spk_cov_df, family=sm.families.Poisson())
results_constant = model_constant.fit()

print(results_constant.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:              1835243
Model:                            GLM   Df Residuals:                  1835242
Model Family:                 Poisson   Df Model:                            0
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -15493.
Date:                Thu, 12 Mar 2026   Deviance:                       27031.
Time:                        13:16:24   Pearson chi2:                 1.83e+06
No. Iterations:                     8   Pseudo R-squ. (CS):              0.000
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -6.8328      0.022   -303.889      0.0

In [12]:
# Interpret the coefficient
mean_count_per_bin = np.exp(results_constant.params["Intercept"])
mean_rate_hz = mean_count_per_bin / BIN_SIZE

print(f"β₀ = {results_constant.params['Intercept']:.4f}")
print(f"exp(β₀) = {mean_count_per_bin:.4f} spikes/bin")
print(f"Firing rate = {mean_rate_hz:.2f} Hz")
print(f"Observed mean = {spk_cov_df['spike_count'].mean():.4f} spikes/bin")

β₀ = -6.8328
exp(β₀) = 0.0011 spikes/bin
Firing rate = 0.54 Hz
Observed mean = 0.0011 spikes/bin


#### Fit on all units

In [ ]:
# null_model_all = fit_glm_all_units("spike_count ~ 1", cov_df_common, spike_counts_common, unit_ids)

In [5]:
# null_model_all["model"] = "null"
# null_model_all.to_csv(f"{base_dir}/analysis/null_model_all.csv")
# keep_default_na=False prevents pandas reading "null" string as NaN
null_model_all = pd.read_csv(f"{base_dir}/analysis/null_model_all.csv", index_col=0,
                              keep_default_na=False, na_values=[''])
null_model_all["model"] = "null"  # re-assign after load in case CSV was saved without it

In [6]:
null_model_all.head()

,unit,aic,llf,deviance,n_params,n_obs,converged,coef,bse,deviance_null,df_model,error,model
0,0,131669.088283882,-65833.544141941,110082.814171104,1.000000000,1756595.000000000,True,{'Intercept': -5.091031019650929},{'Intercept': 0.009619832681734403},110082.814171105,0.000000000,NaN,null
1,1,54608.689862473,-27303.344931237,46950.848745557,1.000000000,1756595.000000000,True,{'Intercept': -6.128267751876933},{'Intercept': 0.016158483907170833},46950.848745556,0.000000000,NaN,null
2,2,26819.161855586,-13408.580927793,23443.161855586,1.000000000,1756595.000000000,True,{'Intercept': -6.948180751506729},{'Intercept': 0.02434681186839675},23443.161855586,0.000000000,NaN,null
3,3,79524.794415936,-39761.397207968,67641.725887742,1.000000000,1756595.000000000,True,{'Intercept': -5.688750247654707},{'Intercept': 0.012970615923499273},67641.725887742,0.000000000,NaN,null
4,4,110086.859673762,-55042.429836881,92622.859673762,1.000000000,1756595.000000000,True,{'Intercept': -5.304252644242502},{'Intercept': 0.010702075387134087},92622.859673762,0.000000000,NaN,null


### Null (outbound only)

#### Fit on one unit

In [15]:
model_constant_out = smf.glm("spike_count ~ 1", 
                             data=spk_cov_df[spk_cov_df["trial_type"]=="outbound"], 
                             family=sm.families.Poisson())
results_constant_out = model_constant_out.fit()

print(results_constant_out.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:               867583
Model:                            GLM   Df Residuals:                   867582
Model Family:                 Poisson   Df Model:                            0
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -4094.8
Date:                Thu, 12 Mar 2026   Deviance:                       7225.7
Time:                        13:16:36   Pearson chi2:                 8.67e+05
No. Iterations:                     9   Pseudo R-squ. (CS):              0.000
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -7.4955      0.046   -164.560      0.0

In [16]:
# Interpret the coefficient
mean_count_per_bin = np.exp(results_constant_out.params["Intercept"])
mean_rate_hz = mean_count_per_bin / BIN_SIZE

print(f"β₀ = {results_constant_out.params['Intercept']:.4f}")
print(f"exp(β₀) = {mean_count_per_bin:.4f} spikes/bin")
print(f"Firing rate = {mean_rate_hz:.2f} Hz")
print(f'Observed mean = {spk_cov_df[spk_cov_df["trial_type"]=="outbound"]["spike_count"].mean():.4f} spikes/bin')

β₀ = -7.4955
exp(β₀) = 0.0006 spikes/bin
Firing rate = 0.28 Hz
Observed mean = 0.0006 spikes/bin


#### Fit on all units

In [ ]:
# null_model_out_all = fit_glm_all_units("spike_count ~ 1", cov_df_out_common, spike_counts_out_common, unit_ids)

In [7]:
# null_model_out_all["model"] = "null_out"
# null_model_out_all.to_csv(f"{base_dir}/analysis/null_model_out_all.csv")
# keep_default_na=False prevents pandas reading "null" string as NaN (fixed)
null_model_out_all = pd.read_csv(f"{base_dir}/analysis/null_model_out_all.csv", index_col=0,
                                  keep_default_na=False, na_values=[''])
null_model_out_all.head(2)

,unit,aic,llf,deviance,n_params,n_obs,converged,coef,bse,deviance_null,df_model,error,model
0,0,56246.726118587,-28122.363059293,47171.043884753,1.000000000,816658.000000000,True,{'Intercept': -5.192073150454998},{'Intercept': 0.014839670194953783},47171.043884753,0.000000000,NaN,null_out
1,1,25383.999939308,-12690.999969654,21826.158822392,1.000000000,816658.000000000,True,{'Intercept': -6.128607038264865},{'Intercept': 0.023702272993155984},21826.158822392,0.000000000,NaN,null_out


### Trial type

#### Fit on one unit

In [18]:
model_trial_type = smf.glm("spike_count ~ trial_type", data=spk_cov_df, family=sm.families.Poisson())
results_trial_type = model_trial_type.fit()

print(results_trial_type.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:              1835243
Model:                            GLM   Df Residuals:                  1835241
Model Family:                 Poisson   Df Model:                            1
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -15273.
Date:                Thu, 12 Mar 2026   Deviance:                       26590.
Time:                        13:16:48   Pearson chi2:                 1.83e+06
No. Iterations:                     9   Pseudo R-squ. (CS):          0.0002400
Covariance Type:            nonrobust                                         
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 -6

In [19]:
rate_inbound  = np.exp(results_trial_type.params["Intercept"]) / BIN_SIZE        # Hz
rate_outbound = np.exp(results_trial_type.params["Intercept"] + results_trial_type.params["trial_type[T.outbound]"]) / BIN_SIZE
ratio         = np.exp(results_trial_type.params["trial_type[T.outbound]"])      # outbound/inbound rate ratio

print("inbound rate: ", rate_inbound)
print("outbound rate: ", rate_outbound)
print("outbound/inbound: ", ratio)

inbound rate:  0.7729987805818047
outbound rate:  0.2777832207690415
outbound/inbound:  0.3593579029451577


#### Fit all units 

In [ ]:
# trial_type_model_all = fit_glm_all_units("spike_count ~ trial_type", cov_df_common, spike_counts_common, unit_ids)

In [8]:
# trial_type_model_all.to_csv(f"{base_dir}/analysis/trial_type_model_all.csv")
trial_type_model_all = pd.read_csv(f"{base_dir}/analysis/trial_type_model_all.csv", index_col=0)
trial_type_model_all["model"] = "trial_type"

In [9]:
trial_type_model_all.head()

,unit,aic,llf,deviance,n_params,n_obs,converged,coef,bse,deviance_null,df_model,error,model
0,0,131583.887559961,-65789.943779981,109995.613447184,2.000000000,1756595.000000000,True,"{'Intercept': -5.010834263411056, 'trial_type[...","{'Intercept': 0.012633958975539225, 'trial_typ...",110082.814171105,1.000000000,NaN,trial_type
1,1,54610.689479532,-27303.344739766,46950.848362615,2.000000000,1756595.000000000,True,"{'Intercept': -6.127973058569825, 'trial_type[...","{'Intercept': 0.022086305064722443, 'trial_typ...",46950.848745556,1.000000000,NaN,trial_type
2,2,26787.700634476,-13391.850317238,23409.700634476,2.000000000,1756595.000000000,True,"{'Intercept': -7.089159110347969, 'trial_type[...","{'Intercept': 0.03571428571380185, 'trial_type...",23443.161855586,1.000000000,NaN,trial_type
3,3,79487.181752602,-39741.590876301,67602.113224407,2.000000000,1756595.000000000,True,"{'Intercept': -5.615587676249746, 'trial_type[...","{'Intercept': 0.01709464148535772, 'trial_type...",67641.725887742,1.000000000,NaN,trial_type
4,4,110046.662126007,-55021.331063003,92580.662126007,2.000000000,1756595.000000000,True,"{'Intercept': -5.241588506345702, 'trial_type[...","{'Intercept': 0.014179049201633688, 'trial_typ...",92622.859673762,1.000000000,NaN,trial_type


### Choice

#### Fit on one unit

In [22]:
choice_mask = cov_df["trial_type"]=="outbound"
choice_spk_cov_df = spk_cov_df[choice_mask]
model_choice= smf.glm("spike_count ~ choice", data=choice_spk_cov_df, family=sm.families.Poisson())
results_choice = model_choice.fit()

print(results_choice.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:               867583
Model:                            GLM   Df Residuals:                   867581
Model Family:                 Poisson   Df Model:                            1
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -4007.2
Date:                Thu, 12 Mar 2026   Deviance:                       7050.5
Time:                        13:17:04   Pearson chi2:                 8.67e+05
No. Iterations:                    10   Pseudo R-squ. (CS):          0.0002019
Covariance Type:            nonrobust                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          -6.4212      0.076    -

In [23]:
rate_left  = np.exp(results_choice.params["Intercept"]) / BIN_SIZE        # Hz
rate_right = np.exp(results_choice.params["Intercept"] + results_choice.params["choice[T.right]"]) / BIN_SIZE
ratio         = np.exp(results_choice.params["choice[T.right]"])      # outbound/inbound rate ratio

print("left rate: ", rate_left)
print("right rate: ", rate_right)
print("right/left: ", ratio)

left rate:  0.8133174791934886
right rate:  0.203945659961309
right/left:  0.2507577485775272


#### Fit for all units

In [ ]:
# choice_model_all = fit_glm_all_units("spike_count ~ choice", cov_df_out_common, spike_counts_out_common, unit_ids)

In [10]:
# choice_model_all.to_csv(f"{base_dir}/analysis/choice_model_all.csv")
choice_model_all = pd.read_csv(f"{base_dir}/analysis/choice_model_all.csv", index_col=0)
choice_model_all["model"] = "choice"

In [11]:
choice_model_all.head()

,unit,aic,llf,deviance,n_params,n_obs,converged,coef,bse,deviance_null,df_model,error,model
0,0,49113.003489998,-24554.501744999,40035.321256165,2.000000000,816658.000000000,True,"{'Intercept': -3.517220133837597, 'choice[T.ri...","{'Intercept': 0.018248288941681203, 'choice[T....",47171.043884753,1.000000000,NaN,choice
1,1,21589.469989400,-10792.734994700,18029.628872483,2.000000000,816658.000000000,True,"{'Intercept': -4.3152469452174795, 'choice[T.r...","{'Intercept': 0.02719641337908737, 'choice[T.r...",21826.158822392,1.000000000,NaN,choice
2,2,11945.222854198,-5970.611427099,10135.222854198,2.000000000,816658.000000000,True,"{'Intercept': -4.942562062926476, 'choice[T.ri...","{'Intercept': 0.03721614637198732, 'choice[T.r...",12293.899149411,1.000000000,NaN,choice
3,3,31548.924988980,-15772.462494490,26503.697577702,2.000000000,816658.000000000,True,"{'Intercept': -4.2839375075651205, 'choice[T.r...","{'Intercept': 0.026773977610566102, 'choice[T....",29160.713401839,1.000000000,NaN,choice
4,4,38792.005071434,-19394.002535717,31274.005071434,2.000000000,816658.000000000,True,"{'Intercept': -3.507939324762412, 'choice[T.ri...","{'Intercept': 0.01816381340601528, 'choice[T.r...",40437.339664736,1.000000000,NaN,choice


### Speed

#### Fit for one unit

In [26]:
spk_cov_df_speed = spk_cov_df_common.copy()  # single-unit speed model (already speed-filtered via common_mask)

model_speed = smf.glm("spike_count ~ speed", data=spk_cov_df_common, family=sm.families.Poisson())
results_speed = model_speed.fit()

print(results_speed.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:              1756595
Model:                            GLM   Df Residuals:                  1756593
Model Family:                 Poisson   Df Model:                            1
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -11508.
Date:                Thu, 12 Mar 2026   Deviance:                       19759.
Time:                        13:17:18   Pearson chi2:                 2.08e+06
No. Iterations:                    10   Pseudo R-squ. (CS):           0.001702
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -9.1519      0.064   -143.726      0.0

In [27]:
#interpret coefficients
beta0_speed = results_speed.params["Intercept"]
beta1_speed = results_speed.params["speed"]

print("Model interpretation:")
print(f"  β₀ = {beta0_speed:.4f} (log rate at speed=0)")
print(f"  β₁ = {beta1_speed:.5f} (change in log rate per cm/s)")
print()
print(f"At speed=0: rate = {np.exp(beta0_speed) / BIN_SIZE:.2f} Hz")
print(f"At speed=20: rate = {np.exp(beta0_speed + beta1_speed * 20) / BIN_SIZE:.2f} Hz")
print(f"Effect: {100 * (np.exp(beta1_speed) - 1):.2f}% change per 1 cm/s")

Model interpretation:
  β₀ = -9.1519 (log rate at speed=0)
  β₁ = 0.04249 (change in log rate per cm/s)

At speed=0: rate = 0.05 Hz
At speed=20: rate = 0.12 Hz
Effect: 4.34% change per 1 cm/s


#### Fit for all units

In [ ]:
# speed_model_all = fit_glm_all_units("spike_count ~ speed", cov_df_common, spike_counts_common, unit_ids)

In [12]:
# speed_model_all["model"] = "speed"
# speed_model_all.to_csv(f"{base_dir}/analysis/speed_model_all.csv")
speed_model_all = pd.read_csv(f"{base_dir}/analysis/speed_model_all.csv", index_col=0)

In [13]:
speed_model_all.head()

,unit,aic,llf,deviance,n_params,n_obs,converged,coef,bse,deviance_null,df_model,error,model
0,0,124504.975695894,-62250.487847947,102916.701583117,2.000000000,1756595.000000000,True,"{'Intercept': -6.188545366071734, 'speed': 0.0...","{'Intercept': 0.01886070291669007, 'speed': 0....",110082.814171105,1.000000000,NaN,speed
1,1,51450.621168929,-25723.310584465,43790.780052013,2.000000000,1756595.000000000,True,"{'Intercept': -7.382680748725404, 'speed': 0.0...","{'Intercept': 0.03294467245533895, 'speed': 0....",46950.848745556,1.000000000,NaN,speed
2,2,25588.375260082,-12792.187630041,22210.375260082,2.000000000,1756595.000000000,True,"{'Intercept': -8.112149264714459, 'speed': 0.0...","{'Intercept': 0.048529188648355634, 'speed': 0...",23443.161855586,1.000000000,NaN,speed
3,3,75495.416870132,-37745.708435066,63610.348341937,2.000000000,1756595.000000000,True,"{'Intercept': -6.801213283177255, 'speed': 0.0...","{'Intercept': 0.025524654325398823, 'speed': 0...",67641.725887742,1.000000000,NaN,speed
4,4,102609.051689684,-51302.525844842,85143.051689684,2.000000000,1756595.000000000,True,"{'Intercept': -6.588488457070062, 'speed': 0.0...","{'Intercept': 0.02198395635135712, 'speed': 0....",92622.859673762,1.000000000,NaN,speed


### Speed (spline)

#### FIt on one unit


In [30]:
from patsy import bs, cr

# speed_min_val, speed_max_val defined in common mask cell
bs_df = 4
model_speed_spline = smf.glm(f"spike_count ~ bs(speed_scaled, df={bs_df})",
                              data=spk_cov_df_common,
                              family=sm.families.Poisson())
results_speed_spline = model_speed_spline.fit()
print(results_speed_spline.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:              1756595
Model:                            GLM   Df Residuals:                  1756590
Model Family:                 Poisson   Df Model:                            4
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -11393.
Date:                Thu, 12 Mar 2026   Deviance:                       19528.
Time:                        13:17:35   Pearson chi2:                 1.90e+06
No. Iterations:                    10   Pseudo R-squ. (CS):           0.001833
Covariance Type:            nonrobust                                         
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept             

In [31]:
speeds_of_interest = [10, 20, 30, 40, 60]
speeds_scaled = (np.array(speeds_of_interest) - speed_min_val) / (speed_max_val - speed_min_val)
rates = results_speed_spline.predict(pd.DataFrame({"speed_scaled": speeds_scaled})) / BIN_SIZE

print("Spline model — predicted rates:")
for s, r in zip(speeds_of_interest, rates):
    print(f"  speed={s:2f} cm/s → {r:.2f} Hz")

speed_range = np.linspace(speed_min_val, speed_max_val, 500)
speed_range_scaled = (speed_range - speed_min_val) / (speed_max_val - speed_min_val)
pred_curve = results_speed_spline.predict(pd.DataFrame({"speed_scaled": speed_range_scaled})) / BIN_SIZE
peak_speed = speed_range[np.argmax(pred_curve)]
print(f"\nPeak firing at: {peak_speed:.1f} cm/s ({pred_curve.max():.2f} Hz)")
print(f"Rate ratio high/low speed: {pred_curve.max() / pred_curve.min():.2f}x")

Spline model — predicted rates:
  speed=10.000000 cm/s → 0.10 Hz
  speed=20.000000 cm/s → 0.07 Hz
  speed=30.000000 cm/s → 0.10 Hz
  speed=40.000000 cm/s → 0.17 Hz
  speed=60.000000 cm/s → 0.55 Hz

Peak firing at: 115.8 cm/s (5.51 Hz)
Rate ratio high/low speed: 89.00x


In [33]:
fig, ax = plt.subplots(figsize=(7, 4))

speed_bins = np.linspace(speed_min_val, speed_max_val, 30)
bin_idx = np.digitize(spk_cov_df_speed["speed"], speed_bins) - 1  # changed

obs_speed, obs_rate, obs_ci = [], [], []
for b in range(len(speed_bins) - 1):
    sel = bin_idx == b
    if sel.sum() > 50:
        counts = spk_cov_df_speed.loc[sel, "spike_count"].values  # changed
        obs_speed.append(speed_bins[b:b+2].mean())
        obs_rate.append(counts.mean() / BIN_SIZE)
        obs_ci.append(1.96 * stats.sem(counts) / BIN_SIZE)

obs_speed, obs_rate, obs_ci = map(np.array, [obs_speed, obs_rate, obs_ci])
ax.errorbar(obs_speed, obs_rate, yerr=obs_ci,
            fmt="o", ms=4, color="grey", ecolor="lightgrey",
            elinewidth=1.5, capsize=3, label="observed ± 95% CI", zorder=3)

speed_range = np.linspace(speed_min_val, speed_max_val, 300)
speed_range_scaled = (speed_range - speed_min_val) / (speed_max_val - speed_min_val)
pred_rate = results_speed_spline.predict(pd.DataFrame({"speed_scaled": speed_range_scaled})) / BIN_SIZE

ax.plot(speed_range, pred_rate, color="steelblue", lw=2, label=f"spline fit (df={bs_df})")
ax.set_xlabel("speed (cm/s)")
ax.set_ylabel("firing rate (Hz)")
ax.set_title(f"unit {unit_idx} — speed tuning")
ax.legend()
sns.despine()
plt.tight_layout()

#### Fit on all units

In [ ]:
# speed_scaled already in cov_df_common (added in common mask cell)
# speed_spline_model_all = fit_glm_all_units(f"spike_count ~ bs(speed_scaled, df={bs_df})",
#                                             cov_df_common, spike_counts_common, unit_ids)

In [14]:
# speed_spline_model_all["model"] = "speed spline"
# speed_spline_model_all.to_csv(f"{base_dir}/analysis/speed_spline_model_all.csv")
speed_spline_model_all = pd.read_csv(f"{base_dir}/analysis/speed_spline_model_all.csv", index_col=0)
speed_spline_model_all.head(1)

,unit,aic,llf,deviance,n_params,n_obs,converged,coef,bse,deviance_null,df_model,error,model
0,0,122885.260660869,-61437.630330435,101290.986548092,5.000000000,1756595.000000000,True,"{'Intercept': -5.013915239708803, 'bs(speed_sc...","{'Intercept': 0.07509759833182705, 'bs(speed_s...",110082.814171105,4.000000000,NaN,speed_spline


### Linear position (spline)

#### Fit on one unit

In [35]:
# pos_min_val, pos_max_val, pos_scaled defined in common mask cell
model_pos_spline = smf.glm("spike_count ~ bs(pos_scaled, df=8)",
                           data=spk_cov_df_common,
                           family=sm.families.Poisson())
results_pos_spline = model_pos_spline.fit()
print(results_pos_spline.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:              1756595
Model:                            GLM   Df Residuals:                  1756586
Model Family:                 Poisson   Df Model:                            8
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -12752.
Date:                Thu, 12 Mar 2026   Deviance:                       22246.
Time:                        13:18:14   Pearson chi2:                 1.75e+06
No. Iterations:                    10   Pseudo R-squ. (CS):          0.0002875
Covariance Type:            nonrobust                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 

In [36]:
pos_range = np.linspace(pos_min_val, pos_max_val, 500)
pos_range_scaled = (pos_range - pos_min_val) / (pos_max_val - pos_min_val)  # changed
pred_rate = results_pos_spline.predict(pd.DataFrame({"pos_scaled": pos_range_scaled})) / BIN_SIZE  # changed

peak_pos  = pos_range[np.argmax(pred_rate)]
peak_rate = pred_rate.max()
print(f"Peak firing position: {peak_pos:.1f} cm, rate: {peak_rate:.2f} Hz")

Peak firing position: 434.1 cm, rate: 1.92 Hz


In [37]:
fig, ax = plt.subplots()

pos_bins = np.linspace(pos_min_val, pos_max_val, 40)
bin_idx  = np.digitize(spk_cov_df["linear_position"].dropna(), pos_bins) - 1
pos_df   = spk_cov_df[spk_cov_df["linear_position"].notna()].copy()

obs_pos, obs_rate, obs_ci = [], [], []
for b in range(len(pos_bins) - 1):
    sel = bin_idx == b
    if sel.sum() > 50:
        counts = pos_df.iloc[np.where(sel)[0]]["spike_count"].values
        obs_pos.append(pos_bins[b:b+2].mean())
        obs_rate.append(counts.mean() / BIN_SIZE)
        obs_ci.append(1.96 * stats.sem(counts) / BIN_SIZE)

ax.errorbar(obs_pos, obs_rate, yerr=obs_ci,
            fmt="o", ms=4, color="grey", ecolor="lightgrey",
            elinewidth=1.5, capsize=3, label="observed ± 95% CI", zorder=3)

ax.plot(pos_range, pred_rate, color="steelblue", lw=2, label="spline fit (df=8)")
ax.axvline(peak_pos, color="red", lw=1, linestyle="--", label=f"peak @ {peak_pos:.0f} cm")
ax.set_xlabel("linear position (cm)")
ax.set_ylabel("firing rate (Hz)")
ax.set_title(f"unit {unit_idx} — position tuning")
ax.legend()
sns.despine()
plt.tight_layout()

In [38]:
graph = sgpl.TrackGraph & {"track_graph_name": "Wtrack_wilbur20210512"}

# Use original camera frames (not 2ms-interpolated bins) — avoids diagonal artifacts
# at trial transitions where np.interp crosses track space (changed)
pos_run = trialized_position[trialized_position["zone"] == "run"][
    ["linear_position", "projected_x_position", "projected_y_position"]
].dropna().iloc[::5]

pos_run = pos_run.copy()
pos_run["pos_scaled"] = (pos_run["linear_position"] - pos_min_val) / (pos_max_val - pos_min_val)
pos_run = pos_run[(pos_run["pos_scaled"] >= 0) & (pos_run["pos_scaled"] <= 1)]

track_rate_hz = results_pos_spline.predict(pos_run[["pos_scaled"]]) / BIN_SIZE

fig, (ax_curve, ax_track) = plt.subplots(1, 2, figsize=(14, 5))

ax_curve.plot(pos_range, pred_rate, color="steelblue", lw=2)
ax_curve.errorbar(obs_pos, obs_rate, yerr=obs_ci,
            fmt="o", ms=4, color="grey", ecolor="lightgrey",
            elinewidth=1.5, capsize=3, label="observed ± 95% CI", zorder=3)
ax_curve.set_xlabel("linear position (cm)")
ax_curve.set_ylabel("firing rate (Hz)")
ax_curve.set_title(f"unit {unit_idx} — position tuning")

graph.plot_track_graph(ax=ax_track, draw_edge_labels=False)
for ln in ax_track.lines:
    ln.set_color("lightgrey")

sc = ax_track.scatter(pos_run["projected_x_position"], pos_run["projected_y_position"],
                      c=track_rate_hz, cmap="hot_r", s=4, zorder=3,
                      vmin=track_rate_hz.min(), vmax=track_rate_hz.max())
plt.colorbar(sc, ax=ax_track, label="firing rate (Hz)")
ax_track.set_xlabel("x position (cm)")
ax_track.set_ylabel("y position (cm)")
ax_track.set_title(f"unit {unit_idx} — rate on track")

sns.despine()
plt.tight_layout()

#### Fit on all units

In [ ]:
# pos_scaled already in cov_df_common (added in common mask cell)
# pos_spline_model_all = fit_glm_all_units("spike_count ~ bs(pos_scaled, df=8)",
#                                           cov_df_common, spike_counts_common, unit_ids)

In [15]:
# pos_spline_model_all["model"] = "pos_spline"
# pos_spline_model_all.to_csv(f"{base_dir}/analysis/pos_spline_model_all.csv")
pos_spline_model_all = pd.read_csv(f"{base_dir}/analysis/pos_spline_model_all.csv", index_col = 0)
pos_spline_model_all.head(2)

,unit,aic,llf,deviance,n_params,n_obs,converged,coef,bse,deviance_null,df_model,error,model
0,0,116512.411290878,-58247.205645439,94910.137178100,9.000000000,1756595.000000000,True,"{'Intercept': -3.5939448356082364, 'bs(pos_sca...","{'Intercept': 0.06155158105423916, 'bs(pos_sca...",110082.814171105,8.000000000,NaN,pos_spline
1,1,51476.254485278,-25729.127242639,43802.413368361,9.000000000,1756595.000000000,True,"{'Intercept': -5.246723861965011, 'bs(pos_scal...","{'Intercept': 0.1273808876721047, 'bs(pos_scal...",46950.848745556,8.000000000,NaN,pos_spline


### Place field maps from pos_spline

In [22]:
import ast
from patsy import dmatrix

# ── shared prediction infrastructure (run once) ───────────────────────────────
pos_pred_vals = np.linspace(pos_min_val, pos_max_val, 300)
pos_pred_scaled = (pos_pred_vals - pos_min_val) / (pos_max_val - pos_min_val)

# Fit reference unit to capture patsy's design_info (knot placement)
_ref = cov_df_common[["pos_scaled"]].copy()
_ref["spike_count"] = spike_counts_common[0]
_ref_fit = smf.glm("spike_count ~ bs(pos_scaled, df=8)", data=_ref, family=sm.families.Poisson()).fit()

X_pred = np.array(dmatrix(_ref_fit.model.data.design_info,
                           pd.DataFrame({"pos_scaled": pos_pred_scaled})))

# Parse stored coefficients — eval() handles bare `nan` tokens that ast.literal_eval rejects
_NAN_NS = {"nan": float("nan")}

def _rate_curve(row):
    if not isinstance(row["coef"], str):
        return np.full(len(pos_pred_vals), np.nan)
    coefs = np.array(list(eval(row["coef"], _NAN_NS).values()))
    if np.any(np.isnan(coefs)):
        return np.full(len(pos_pred_vals), np.nan)
    return np.exp(X_pred @ coefs) / BIN_SIZE

rate_matrix = np.vstack([_rate_curve(r) for _, r in pos_spline_model_all.iterrows()])

# Peak position — NaN for units whose fit failed (all-NaN row)
valid_units = ~np.isnan(rate_matrix).all(axis=1)
peak_pos_cm = np.full(len(rate_matrix), np.nan)
peak_pos_cm[valid_units] = pos_pred_vals[np.argmax(rate_matrix[valid_units], axis=1)]

# Track projection data (downsampled) — used by all track plots
pos_run = trialized_position[trialized_position["zone"] == "run"][
    ["linear_position", "projected_x_position", "projected_y_position"]
].dropna().iloc[::5].copy()
pos_run["pos_scaled"] = (pos_run["linear_position"] - pos_min_val) / (pos_max_val - pos_min_val)
pos_run = pos_run[(pos_run["pos_scaled"] >= 0) & (pos_run["pos_scaled"] <= 1)]

n_failed = (~valid_units).sum()
print(f"rate_matrix: {rate_matrix.shape}  ({n_failed} units failed/skipped)")
print(f"peak positions range: {np.nanmin(peak_pos_cm):.0f} – {np.nanmax(peak_pos_cm):.0f} cm")

rate_matrix: (263, 300)  (2 units failed/skipped)
peak positions range: 24 – 504 cm


#### 1. Single unit

In [23]:
def plot_place_field(uid, ax_curve=None, ax_track=None, graph=None):
    """Rate curve + track heatmap for one unit. Pass axes to embed in a grid."""
    rate = rate_matrix[uid]
    standalone = ax_curve is None
    if standalone:
        fig, (ax_curve, ax_track) = plt.subplots(1, 2, figsize=(13, 4))

    # ── rate curve ────────────────────────────────────────────────────────────
    ax_curve.plot(pos_pred_vals, rate, color="steelblue", lw=1.5)
    ax_curve.axvline(peak_pos_cm[uid], color="red", lw=1, ls="--",
                     label=f"peak @ {peak_pos_cm[uid]:.0f} cm")
    ax_curve.set_xlabel("linear position (cm)")
    ax_curve.set_ylabel("firing rate (Hz)")
    ax_curve.set_title(f"unit {uid}")
    ax_curve.legend(fontsize=8)
    sns.despine(ax=ax_curve)

    # ── track heatmap ─────────────────────────────────────────────────────────
    track_rate = np.interp(pos_run["linear_position"], pos_pred_vals, rate)
    if graph is None:
        graph = sgpl.TrackGraph & {"track_graph_name": "Wtrack_wilbur20210512"}
    graph.plot_track_graph(ax=ax_track, draw_edge_labels=False)
    for ln in ax_track.lines:
        ln.set_color("lightgrey")
    sc = ax_track.scatter(pos_run["projected_x_position"], pos_run["projected_y_position"],
                          c=track_rate, cmap="hot_r", s=3, zorder=3,
                          vmin=rate.min(), vmax=rate.max())
    plt.colorbar(sc, ax=ax_track, label="Hz", shrink=0.8)
    ax_track.set_title(f"unit {uid} — peak {peak_pos_cm[uid]:.0f} cm")
    ax_track.set_xlabel("x (cm)"); ax_track.set_ylabel("y (cm)")

    if standalone:
        plt.tight_layout()

# ── select unit ───────────────────────────────────────────────────────────────
selected_unit = 12
plot_place_field(selected_unit)

#### 2. Grid of selected units

In [24]:
def plot_place_field_grid(unit_list, ncols=4):
    """Track heatmaps for a list of units arranged in a grid."""
    n = len(unit_list)
    nrows = int(np.ceil(n / ncols))
    graph = sgpl.TrackGraph & {"track_graph_name": "Wtrack_wilbur20210512"}

    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.5))
    axes = np.array(axes).flatten()

    for ax, uid in zip(axes, unit_list):
        rate = rate_matrix[uid]
        track_rate = np.interp(pos_run["linear_position"], pos_pred_vals, rate)
        graph.plot_track_graph(ax=ax, draw_edge_labels=False)
        for ln in ax.lines:
            ln.set_color("lightgrey")
        sc = ax.scatter(pos_run["projected_x_position"], pos_run["projected_y_position"],
                        c=track_rate, cmap="hot_r", s=2, zorder=3,
                        vmin=rate.min(), vmax=rate.max())
        plt.colorbar(sc, ax=ax, label="Hz", shrink=0.7)
        ax.set_title(f"unit {uid}  peak={peak_pos_cm[uid]:.0f}cm", fontsize=9)
        ax.set_xlabel(""); ax.set_ylabel("")

    for ax in axes[n:]:   # hide unused panels
        ax.set_visible(False)

    plt.tight_layout()

# ── select units ──────────────────────────────────────────────────────────────
grid_units = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
plot_place_field_grid(grid_units, ncols=4)

#### 3. Population summary (all units)

In [26]:
# ── population heatmap: valid units only, sorted by peak position ─────────────
valid_idx = np.where(valid_units)[0]
sort_order = valid_idx[np.argsort(peak_pos_cm[valid_idx])]

rate_sorted = rate_matrix[sort_order]
rate_norm = (rate_sorted - rate_sorted.min(axis=1, keepdims=True)) / \
            (rate_sorted.max(axis=1, keepdims=True) - rate_sorted.min(axis=1, keepdims=True) + 1e-9)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im = axes[0].imshow(rate_norm, aspect="auto", origin="lower", cmap="hot_r",
                    extent=[pos_pred_vals[0], pos_pred_vals[-1], 0, len(sort_order)])
plt.colorbar(im, ax=axes[0], label="normalised rate")
axes[0].set_xlabel("linear position (cm)")
axes[0].set_ylabel(f"unit (sorted by peak,  n={len(sort_order)})")
axes[0].set_title("Population place fields")

axes[1].hist(peak_pos_cm[valid_idx], bins=30, color="steelblue", edgecolor="white")
axes[1].set_xlabel("peak position (cm)")
axes[1].set_ylabel("unit count")
axes[1].set_title("Distribution of place field peaks")
sns.despine(ax=axes[1])

plt.tight_layout()

### Trial progress (spline)

#### Fit on one unit

In [70]:
tp_df = 6

# trial_progress is already [0, 1] — no scaling needed
print(f"trial_progress NaN in cov_df_common: {spk_cov_df_common['trial_progress'].isna().sum()}")

model_tp_spline = smf.glm(
    f"spike_count ~ cr(trial_progress, df={tp_df}, constraints='center')",
    data=spk_cov_df_common,
    family=sm.families.Poisson()
)
results_tp_spline = model_tp_spline.fit()
print(results_tp_spline.summary())

trial_progress NaN in cov_df_common: 0
                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:              1756595
Model:                            GLM   Df Residuals:                  1756588
Model Family:                 Poisson   Df Model:                            6
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -11665.
Date:                Thu, 12 Mar 2026   Deviance:                       20071.
Time:                        16:20:29   Pearson chi2:                 1.46e+06
No. Iterations:                   100   Pseudo R-squ. (CS):           0.001525
Covariance Type:            nonrobust                                         
                                  coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------

In [71]:
tp_min = spk_cov_df_common["trial_progress"].min()
tp_max = spk_cov_df_common["trial_progress"].max()
tp_range = np.linspace(tp_min, tp_max, 300)
pred_tp = results_tp_spline.predict(pd.DataFrame({"trial_progress": tp_range})) / BIN_SIZE

peak_tp = tp_range[np.argmax(pred_tp)]
print(f"Peak firing at trial_progress = {peak_tp:.2f}, rate = {pred_tp.max():.2f} Hz")

# Observed binned rate
tp_bins = np.linspace(tp_min, tp_max, 25)
tp_bin_idx = np.digitize(spk_cov_df_common["trial_progress"], tp_bins) - 1
obs_tp, obs_tp_rate, obs_tp_ci = [], [], []
for b in range(len(tp_bins) - 1):
    sel = tp_bin_idx == b
    if sel.sum() > 50:
        counts = spk_cov_df_common.loc[sel, "spike_count"].values
        obs_tp.append(tp_bins[b:b+2].mean())
        obs_tp_rate.append(counts.mean() / BIN_SIZE)
        obs_tp_ci.append(1.96 * stats.sem(counts) / BIN_SIZE)

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(obs_tp, obs_tp_rate, yerr=obs_tp_ci,
            fmt="o", ms=4, color="grey", ecolor="lightgrey",
            elinewidth=1.5, capsize=3, label="observed ± 95% CI", zorder=3)
ax.plot(tp_range, pred_tp, color="steelblue", lw=2, label=f"spline fit (df={tp_df})")
ax.axvline(peak_tp, color="red", lw=1, linestyle="--", label=f"peak @ {peak_tp:.2f}")
ax.set_xlabel("trial progress (0 = start, 1 = end)")
ax.set_ylabel("firing rate (Hz)")
ax.set_title(f"unit {unit_idx} — trial progress tuning")
ax.legend()
sns.despine()
plt.tight_layout()

Peak firing at trial_progress = 1.00, rate = 4.15 Hz


#### Fit on all units

In [ ]:
# tp_df defined above
# tp_progress_spline_model_all = fit_glm_all_units(f"spike_count ~ cr(trial_progress, df={tp_df}, constraints='center')",
#                                                   cov_df_common, spike_counts_common, unit_ids)

In [27]:
# tp_progress_spline_model_all["model"] = "trial_progress_spline"
# tp_progress_spline_model_all.to_csv(f"{base_dir}/analysis/trial_progress_spline_model_all.csv")
trial_progress_spline_model_all = pd.read_csv(
    f"{base_dir}/analysis/trial_progress_spline_model_all.csv", index_col=0
)
trial_progress_spline_model_all.head(2)

,unit,aic,llf,deviance,n_params,n_obs,converged,coef,bse,deviance_null,df_model,error,model
0,0,125111.039538532,-62548.519769266,103512.765425755,7.000000000,1756595.000000000,True,"{'Intercept': -5.935167341177781, ""cr(trial_pr...","{'Intercept': 0.05217593190562265, ""cr(trial_p...",110082.814171105,6.000000000,NaN,trial_progress_spline
1,1,51926.193294323,-25956.096647162,44256.352177407,7.000000000,1756595.000000000,True,"{'Intercept': -7.192475542188085, ""cr(trial_pr...","{'Intercept': 0.1273311312072992, ""cr(trial_pr...",46950.848745556,6.000000000,NaN,trial_progress_spline


## Model comparison

In [28]:
from scipy.stats import chi2

# Load all model CSVs
model_files = {
    "null":                    (f"{base_dir}/analysis/null_model_all.csv",                    dict(keep_default_na=False, na_values=[""])),
    "null_out":                (f"{base_dir}/analysis/null_model_out_all.csv",                dict(keep_default_na=False, na_values=[""])),
    "trial_type":              (f"{base_dir}/analysis/trial_type_model_all.csv",              {}),
    "choice":                  (f"{base_dir}/analysis/choice_model_all.csv",                  {}),
    "speed":                   (f"{base_dir}/analysis/speed_model_all.csv",                   {}),
    "speed_spline":            (f"{base_dir}/analysis/speed_spline_model_all.csv",            {}),
    "pos_spline":              (f"{base_dir}/analysis/pos_spline_model_all.csv",              {}),
    "trial_progress_spline":   (f"{base_dir}/analysis/trial_progress_spline_model_all.csv",  {}),
}

models = {}
for name, (path, kwargs) in model_files.items():
    df = pd.read_csv(path, index_col=0, **kwargs)
    df["model"] = name
    models[name] = df.set_index("unit")

# Null baseline per model — must match the dataset the model was fit on
null_for = {
    "trial_type":            "null",
    "speed":                 "null",
    "speed_spline":          "null",
    "pos_spline":            "null",
    "trial_progress_spline": "null",
    "choice":                "null_out",
}

rows = []
for model_name, null_name in null_for.items():
    m   = models[model_name]
    nul = models[null_name]

    for uid in m.index:
        row      = m.loc[uid]
        null_row = nul.loc[uid]

        lrt_stat = 2 * (row["llf"] - null_row["llf"])
        lrt_df   = int(row["df_model"]) if pd.notna(row["df_model"]) else 0
        lrt_pval = (1 - chi2.cdf(lrt_stat, lrt_df)) if lrt_df > 0 else np.nan

        rows.append(dict(
            unit        = uid,
            model       = model_name,
            aic         = row["aic"],
            llf         = row["llf"],
            n_params    = row["n_params"],
            n_obs       = row["n_obs"],
            converged   = row["converged"],
            delta_aic   = row["aic"] - null_row["aic"],
            lrt_stat    = lrt_stat,
            lrt_df      = lrt_df,
            lrt_pval    = lrt_pval,
            tuned       = bool(lrt_pval < 0.05) if not np.isnan(lrt_pval) else False,
        ))

comparison = pd.DataFrame(rows)

summary = comparison.groupby("model").agg(
    mean_dAIC  = ("delta_aic", "mean"),
    median_dAIC= ("delta_aic", "median"),
    n_tuned    = ("tuned",     "sum"),
    frac_tuned = ("tuned",     "mean"),
    n_converged= ("converged", "sum"),
).round(3)
print(summary)

                            mean_dAIC     median_dAIC  n_tuned  frac_tuned  \
model                                                                        
choice                -1312.531000000  -571.002000000      252 0.958000000   
pos_spline            -3523.824000000 -1921.845000000      261 0.992000000   
speed                 -3061.827000000 -1593.043000000      257 0.977000000   
speed_spline          -3592.272000000 -1922.189000000      262 0.996000000   
trial_progress_spline -2641.916000000 -1540.833000000      260 0.989000000   
trial_type              -55.532000000   -15.324000000      191 0.726000000   

                       n_converged  
model                               
choice                         262  
pos_spline                     256  
speed                          262  
speed_spline                   261  
trial_progress_spline          233  
trial_type                     262  


In [29]:
# Pivot to wide AIC table — enables direct pairwise model comparison
aic_wide = comparison.pivot(index="unit", columns="model", values="aic")

# Add null baselines (same common dataset, so directly comparable)
aic_wide["null"]     = models["null"]["aic"]
aic_wide["null_out"] = models["null_out"]["aic"]

# Best single-variable model per unit (all-trials models only)
all_trials_models = ["trial_type", "speed", "speed_spline", "pos_spline", "trial_progress_spline"]
aic_wide["best_model"] = aic_wide[all_trials_models].idxmin(axis=1)
print("Best model counts (all units):")
print(aic_wide["best_model"].value_counts())

# LRT: speed_linear vs speed_spline (nested — spline adds 3 df)
llf_wide = comparison.pivot(index="unit", columns="model", values="llf")
lrt_speed = 2 * (llf_wide["speed_spline"] - llf_wide["speed"])
lrt_speed_pval = lrt_speed.apply(lambda x: 1 - chi2.cdf(x, df=3))  # Δdf = 5-2 = 3
n_nonlinear = (lrt_speed_pval < 0.05).sum()
print(f"\nSpeed: nonlinear > linear in {n_nonlinear}/{len(lrt_speed_pval)} units (LRT p<0.05)")

Best model counts (all units):
best_model
speed_spline             133
pos_spline               122
trial_progress_spline      7
Name: count, dtype: int64

Speed: nonlinear > linear in 258/263 units (LRT p<0.05)


/tmp/ipykernel_367601/3622809798.py:10: FutureWarning: The behavior of DataFrame.idxmin with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  aic_wide["best_model"] = aic_wide[all_trials_models].idxmin(axis=1)


In [31]:
# ── Q1: what fraction of units are significantly tuned to each variable? ───────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

frac = summary["frac_tuned"].sort_values()
frac.plot.barh(ax=axes[0], color="steelblue")
axes[0].axvline(0.05, color="red", lw=1, ls="--", label="chance (α=0.05)")
axes[0].set_xlabel("fraction of units (LRT p < 0.05)")
axes[0].set_title("Fraction tuned per variable")
axes[0].legend(fontsize=9)
sns.despine(ax=axes[0])

# ── Q2: are different units tuned to different variables? ──────────────────────
# Pivot to unit × model binary matrix
tuned_wide = comparison.pivot(index="unit", columns="model", values="tuned").astype(int)

# Sort rows: units tuned to more variables go to the top
row_order = tuned_wide.sum(axis=1).sort_values().index
tuned_sorted = tuned_wide.loc[row_order]

axes[1].imshow(tuned_sorted.values, aspect="auto", cmap="Blues",
               interpolation="nearest", vmin=0, vmax=1)
axes[1].set_xticks(range(len(tuned_wide.columns)))
axes[1].set_xticklabels(tuned_wide.columns, rotation=40, ha="right", fontsize=9)
axes[1].set_ylabel("unit (sorted by n tuned variables)")
axes[1].set_xlabel("model")
axes[1].set_title("Tuning profile per unit")

plt.tight_layout()

In [76]:
aic_wide.head(2)

model,choice,pos_spline,speed,speed_spline,trial_progress_spline,trial_type,null,null_out,best_model
unit,,,,,,,,,
0,49113.003489998,116512.411290878,124504.975695894,122885.260660869,125069.774208753,131583.887559961,131669.088283882,56246.726118587,pos_spline
1,21589.469989400,51476.254485278,51450.621168929,51005.473623726,51905.169874022,54610.689479532,54608.689862473,25383.999939308,speed_spline


In [32]:
# which model has the highest mean delta-AIC?
model_cols = ["trial_type", "speed", "speed_spline", "pos_spline", "trial_progress_spline"]
delta_aic = aic_wide[model_cols].subtract(aic_wide["null"], axis = 0)
print(delta_aic.mean(axis=0))
print("model with max delta aic: ", min(delta_aic.mean(axis = 0)))

model
trial_type                -55.532004209
speed                   -3061.826646431
speed_spline            -3592.271583448
pos_spline              -3523.823842328
trial_progress_spline   -2641.916287258
dtype: float64
model with max delta aic:  -3592.271583447503


## Combined models

### Full model (trial type, position , speed)

#### Fit on one unit

In [94]:
full_model_formula = "spike_count ~ trial_type + bs(pos_scaled, df = 8) + bs(speed_scaled, df = 4)"


In [95]:
model_full_pos = smf.glm(
    full_model_formula,
    data=spk_cov_df_common,
    family=sm.families.Poisson()
)
results_full_pos = model_full_pos.fit(maxiter = 200)
print(results_full_pos.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:              1756595
Model:                            GLM   Df Residuals:                  1756581
Model Family:                 Poisson   Df Model:                           13
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -10877.
Date:                Thu, 12 Mar 2026   Deviance:                       18495.
Time:                        17:41:12   Pearson chi2:                 1.65e+06
No. Iterations:                    11   Pseudo R-squ. (CS):           0.002420
Covariance Type:            nonrobust                                         
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept             

#### Fit on all units

In [ ]:
# full_model_all = fit_glm_all_units(full_model_formula, cov_df_common, spike_counts_common, unit_ids)

In [34]:
# full_model_all["model"] = "full_model"
# full_model_all.to_csv(f"{base_dir}/analysis/full_model_all.csv")
full_model_all = pd.read_csv(f"{base_dir}/analysis/full_model_all.csv", index_col=0)
full_model_all.head(2)

,unit,aic,llf,deviance,n_params,n_obs,converged,coef,bse,deviance_null,df_model,error,model
0,0,114114.147088039,-57043.073544019,92501.872975261,14.000000000,1756595.000000000,True,"{'Intercept': -3.629033564726252, 'trial_type[...","{'Intercept': 0.0928687142925612, 'trial_type[...",110082.814171105,13.000000000,NaN,full_model
1,1,48964.875886940,-24468.437943470,41281.034770024,14.000000000,1756595.000000000,True,"{'Intercept': -4.79925523623829, 'trial_type[T...","{'Intercept': 0.1689481665382055, 'trial_type[...",46950.848745556,13.000000000,NaN,full_model


### Temporal full model (trial type, progress, speed)

Replaces position with trial_progress — same structure as full model but asks whether mPFC tracks *when within a trial* rather than *where on the track*. Compare AIC directly to `full_model` (same dataset, same n_obs).

#### Fit on one unit

In [104]:
temporal_model_formula = (
    f"spike_count ~ trial_type  + cr(trial_progress, df={tp_df}, constraints='center') + bs(speed_scaled, df=4)"
)

model_temporal = smf.glm(temporal_model_formula, data=spk_cov_df_common, family=sm.families.Poisson())
results_temporal = model_temporal.fit(disp=False)
print(results_temporal.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            spike_count   No. Observations:              1756595
Model:                            GLM   Df Residuals:                  1756583
Model Family:                 Poisson   Df Model:                           11
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -10732.
Date:                Thu, 12 Mar 2026   Deviance:                       18205.
Time:                        18:14:50   Pearson chi2:                 2.05e+06
No. Iterations:                    13   Pseudo R-squ. (CS):           0.002584
Covariance Type:            nonrobust                                         
                                                        coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------

#### Fit on all units

In [ ]:
# temporal_model_all = fit_glm_all_units(temporal_model_formula, cov_df_common, spike_counts_common, unit_ids)

In [35]:
# temporal_model_all["model"] = "temporal_model"
# temporal_model_all.to_csv(f"{base_dir}/analysis/temporal_model_all.csv")
temporal_model_all = pd.read_csv(f"{base_dir}/analysis/temporal_model_all.csv", index_col=0)
temporal_model_all.head(2)

,unit,aic,llf,deviance,n_params,n_obs,converged,coef,bse,deviance_null,df_model,error,model
0,0,118032.990517666,-59004.495258833,96424.716404889,12.000000000,1756595.000000000,True,"{'Intercept': -3.4491169200348617, 'trial_type...","{'Intercept': 0.07673825433941116, 'trial_type...",110082.814171105,11.000000000,NaN,temporal_model
1,1,49719.999530268,-24847.999765134,42040.158413352,12.000000000,1756595.000000000,True,"{'Intercept': -4.504202408649432, 'trial_type[...","{'Intercept': 0.1508563395616019, 'trial_type[...",46950.848745556,11.000000000,NaN,temporal_model
